In [8]:
import tiktoken
import pandas as pd
import json

In [9]:
import openai
from openai import OpenAI

openai.api_key = 'sk-proj-YLvCCJmFWMusZAH0wYuAT3BlbkFJzuOeCfODAfNJAaAONKg6'
client = OpenAI(api_key=openai.api_key)

In [10]:
encoding = tiktoken.get_encoding("cl100k_base")

In [11]:
def count_tokens(text):
    return len(encoding.encode(text))

def batch_entries(entries, max_tokens):
    batches = []
    current_batch = []
    current_tokens = 0

    for entry in entries:
        entry_tokens = count_tokens(entry)

        if current_tokens + entry_tokens + 1 > max_tokens:
            batches.append(" ".join(current_batch))  
            current_batch = [entry]
            current_tokens = entry_tokens
        else:
            current_batch.append(entry)
            current_tokens += entry_tokens + 1 

    if current_batch:
        batches.append(" ".join(current_batch))

    return batches

def get_embedding(batch, model="text-embedding-3-small"):
   batch = batch.replace("\n", " ")
   return client.embeddings.create(input=[batch], model=model).data[0].embedding

def process_text(entries, max_tokens=8191):
    batches = batch_entries(entries, max_tokens)
    embeddings = [get_embedding(batch) for batch in batches]
    return embeddings

In [12]:
def process_and_save_embeddings(input_file, csv_file, json_file, max_tokens=8191, batch_size=1000):
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    batch_count = 0
    temp_progress = []

    # Initialize or read the existing JSON file
    try:
        with open(json_file, 'r', encoding='utf-8') as jf:
            existing_json_data = json.load(jf)
    except (FileNotFoundError, json.JSONDecodeError):
        existing_json_data = []

    # Process in batches
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        batch_text = [json.dumps(entry) for entry in batch]
        embeddings = process_text(batch_text, max_tokens=max_tokens)

        for entry, embedding in zip(batch, embeddings):
            entry['embedding'] = embedding
            temp_progress.append(entry)
        
        # Append to JSON
        existing_json_data.extend(temp_progress)
        
        temp_progress = []
        batch_count += 1
        print(f"Processed batch {batch_count}")
        
    with open(json_file, 'w', encoding='utf-8') as jf:
            json.dump(existing_json_data, jf, ensure_ascii=False, indent=2)
    print(f"Processed all entries: {len(data)}")

In [13]:
input_file = 'apuracao_icms_a_pagar_202407142339.json'
json_file = 'embeddings-' + input_file
csv_file = json_file.replace('.json', '.csv')

In [14]:
process_and_save_embeddings(input_file, json_file, csv_file)

Processed batch 1
Processed all entries: 48
